# Week 02A — Image Formation, Optics and Lighting

**MCTA 4364 Machine Vision** · Session 1 of Week 2 · 80 minutes

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W02_optics_calibration/W02A_image_formation_optics_lighting.ipynb) [![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/hasanzaki/MCTA-4364-Machine-Vision/blob/main/notebooks/W02_optics_calibration/W02A_image_formation_optics_lighting.ipynb)

Before any algorithm runs, three physical components decide what the image can show: the **camera geometry**, the **lens** and
the **lighting**. Experienced machine-vision engineers spend most of their design time here, because a good image makes the
software simple and a bad image makes it impossible. This session builds the mathematical camera model you will calibrate in
Session B and use again for measurement (Week 10) and 3D vision (Week 12).

### Learning outcomes
By the end of this session you will be able to:
1. Derive the **pinhole projection** and write it with the intrinsic matrix $K$ and extrinsics $[R\,|\,t]$ in homogeneous coordinates.
2. Explain perspective effects: size ∝ 1/depth, vanishing points.
3. Use the **thin-lens equation**, **field of view** and **depth of field** to select a lens for an inspection task.
4. Model and visualise **radial and tangential distortion**, **vignetting** and **telecentric** optics.
5. Choose a **lighting technique** (bright-field, dark-field, dome, backlight, colour) to make a defect visible.

### Session plan (80 min)
| Time | Section | Activity |
|---|---|---|
| 0–20 | 1–2 | Pinhole camera, perspective, K and [R\|t] (interactive 3D) |
| 20–35 | 3–4 | Lenses, depth of field, choosing a lens (calculators) |
| 35–45 | 5–6 | Distortion, vignetting, telecentric lenses |
| 45–70 | 7 | Lighting for machine vision (interactive lighting simulator) |
| 70–80 | 10 | Quiz and exit ticket |
| Home | 8–9 | Exercises with self-checks, system-design challenge |

**Prerequisites:** Week 01 (images as arrays), basic trigonometry and matrices.

## 0. Setup

In [ ]:
# Setup: works in Google Colab, Kaggle, Jupyter/VS Code on your laptop, and CI.
import importlib.util
import os
import subprocess
import sys

REPO_URL = "https://github.com/hasanzaki/MCTA-4364-Machine-Vision.git"
if not os.path.exists("resources/scripts/cvhelpers.py"):
    if os.path.exists("../../resources/scripts/cvhelpers.py"):  # opened from its own folder
        os.chdir("../..")
    else:  # Colab / Kaggle: fetch the course helpers
        if not os.path.isdir("MCTA-4364-Machine-Vision"):
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
        os.chdir("MCTA-4364-Machine-Vision")
print("Working directory:", os.getcwd())

# Install only what is missing. PyTorch is deliberately NOT pip-installed here:
# that would replace your laptop's CUDA build with a CPU-only one (see SETUP.md).
needed = {"cv2": "opencv-python", "skimage": "scikit-image", "ipywidgets": "ipywidgets",
          "matplotlib": "matplotlib", "scipy": "scipy", "plotly": "plotly"}
missing = [pkg for mod, pkg in needed.items() if importlib.util.find_spec(mod) is None]
if missing:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
sys.path.append("resources/scripts")

In [ ]:
import cv2
import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import plotly.graph_objects as go

from cvhelpers import check, interact, load_sample, quiz, show

plt.rcParams.update({"figure.dpi": 100, "axes.titlesize": 11})

## 1. The pinhole camera

Light from a scene point passes through a tiny hole and hits the image plane at distance $f$ behind it. By **similar triangles**:

$$ x = f\,\frac{X}{Z}, \qquad y = f\,\frac{Y}{Z} $$

where $(X, Y, Z)$ is the point in the **camera frame** (Z along the optical axis) and $(x, y)$ is its position on the image plane.
The real image is upside-down; mathematically we use a *virtual* image plane in front of the pinhole, which gives the same equations
without the sign flip. The division by $Z$ is the heart of **perspective**: objects twice as far away appear half as large.

In [ ]:
fig, ax = plt.subplots(figsize=(15, 4.8))
Z_obj, H_obj, f = 8.0, 2.0, 3.0
ax.annotate("", xy=(-Z_obj, H_obj), xytext=(-Z_obj, 0), arrowprops=dict(arrowstyle="-|>", color="#188038", lw=3))
ax.text(-Z_obj - 0.3, H_obj / 2, "object\nheight Y", ha="right", va="center", color="#188038")
ax.plot([-Z_obj, f], [H_obj, -H_obj * f / Z_obj], color="#fbbc04", lw=1.5)
ax.plot([-Z_obj, f], [0, 0], color="#9aa0a6", lw=1, ls="--")
ax.annotate("", xy=(f, -H_obj * f / Z_obj), xytext=(f, 0), arrowprops=dict(arrowstyle="-|>", color="#d93025", lw=3))
ax.annotate("", xy=(-f, H_obj * f / Z_obj), xytext=(-f, 0), arrowprops=dict(arrowstyle="-|>", color="#d93025", lw=2, ls="--"))
ax.plot([f, f], [-1.5, 1.5], color="k", lw=2); ax.text(f + 0.1, 1.4, "real image plane\n(inverted)", fontsize=9)
ax.plot([-f, -f], [-1.5, 1.5], color="k", lw=1, ls=":"); ax.text(-f - 0.1, -1.4, "virtual image plane\n(used in the maths)", fontsize=9, ha="right")
ax.plot([0, 0], [0.12, 1.8], color="k", lw=5); ax.plot([0, 0], [-0.12, -1.8], color="k", lw=5)
ax.text(0.1, 1.9, "pinhole (optical centre)", fontsize=9)
ax.annotate("", xy=(0, -2.2), xytext=(-Z_obj, -2.2), arrowprops=dict(arrowstyle="<->")); ax.text(-Z_obj / 2, -2.45, "depth Z", ha="center")
ax.annotate("", xy=(f, -2.2), xytext=(0, -2.2), arrowprops=dict(arrowstyle="<->")); ax.text(f / 2, -2.45, "focal length f", ha="center")
ax.set_xlim(-10.5, 5); ax.set_ylim(-2.8, 2.6); ax.axis("off")
ax.set_title("Figure 1.1 - Pinhole camera: y = f Y / Z (similar triangles)", fontweight="bold")
plt.show()

### 1.1 From metres to pixels: the intrinsic matrix $K$
The sensor samples the image plane into pixels of size $s_x \times s_y$ (e.g. 3.45 µm). Expressing $f$ in **pixels** ($f_x = f/s_x$)
and moving the origin to the top-left corner (the **principal point** $(c_x, c_y)$ is where the optical axis hits the sensor) gives,
in **homogeneous coordinates**:

$$ \lambda \begin{bmatrix} u \\ v \\ 1 \end{bmatrix} = \underbrace{\begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}}_{K\ \text{(intrinsics)}}
\underbrace{\begin{bmatrix} R & \mathbf{t} \end{bmatrix}}_{\text{extrinsics}} \begin{bmatrix} X_w \\ Y_w \\ Z_w \\ 1 \end{bmatrix} $$

- $[R\,|\,\mathbf{t}]$ moves a world point into the camera frame (rotation, then translation): the camera's **pose**.
- $K$ projects camera coordinates to pixels. The scale $\lambda$ equals the depth $Z$; dividing by it performs the perspective division.
- Homogeneous coordinates turn the non-linear division into one matrix product, which is why they are used throughout vision.

In [ ]:
# Figure 1.2: perspective effects on a synthetic scene
W, H = 640, 480
K = np.array([[500, 0, W / 2], [0, 500, H / 2], [0, 0, 1]], float)


def project(Pc):
    """Pinhole projection of camera-frame points (N x 3) to pixels (N x 2)."""
    p = (K @ Pc.T).T
    return p[:, :2] / p[:, 2:3]


fig, axes = plt.subplots(1, 2, figsize=(16, 5.6))
ax = axes[0]
for x in np.arange(-3, 3.01, 0.5):                      # floor lines, 1.2 m below the camera
    seg = project(np.array([[x, 1.2, 1.5], [x, 1.2, 30]]))
    ax.plot(seg[:, 0], seg[:, 1], color="#1a73e8", lw=1)
for z in np.arange(1.5, 30, 1.5):
    seg = project(np.array([[-3, 1.2, z], [3, 1.2, z]]))
    ax.plot(seg[:, 0], seg[:, 1], color="#1a73e8", lw=0.6, alpha=0.6)
for z, col in [(3, "#d93025"), (6, "#e37400"), (12, "#188038")]:   # identical 1 m poles at different depths
    seg = project(np.array([[1.0, 1.2, z], [1.0, 0.2, z]]))
    ax.plot(seg[:, 0], seg[:, 1], color=col, lw=5, label=f"1 m pole at Z = {z} m: {abs(seg[0, 1] - seg[1, 1]):.0f} px")
ax.plot(W / 2, H / 2, "k*", ms=14); ax.text(W / 2 + 8, H / 2 - 8, "vanishing point")
ax.set_xlim(0, W); ax.set_ylim(H, 0); ax.set_aspect("equal"); ax.legend(loc="lower left", fontsize=8)
ax.set_title("parallel floor lines meet at a vanishing point; size ~ 1/Z")
Z = np.linspace(0.5, 12, 200)
axes[1].plot(Z, 500 * 1.0 / Z, lw=2.5)
axes[1].set_xlabel("depth Z (m)"); axes[1].set_ylabel("image height of a 1 m object (px)"); axes[1].grid(alpha=0.3)
axes[1].set_title("apparent size = f * size / Z")
plt.suptitle("Figure 1.2 - Perspective projection", fontweight="bold")
plt.tight_layout(); plt.show()

## 2. Interactive: move the camera

The cube below is 1 m wide and sits 5 m in front of the world origin. Change the focal length and the camera pose and watch the
projection. Notice the difference between **zooming** (changing $f$: the image scales, perspective stays the same) and
**moving closer** (changing $t_z$: perspective distortion grows). Photographers call this the *dolly zoom*.

In [ ]:
cube = np.array([[x, y, z] for x in (-0.5, 0.5) for y in (-0.5, 0.5) for z in (4.5, 5.5)], float)
edges = [(i, j) for i in range(8) for j in range(i + 1, 8) if np.sum(np.abs(cube[i] - cube[j]) > 0) == 1]


def rot_y(deg):
    a = np.deg2rad(deg)
    return np.array([[np.cos(a), 0, np.sin(a)], [0, 1, 0], [-np.sin(a), 0, np.cos(a)]])


def camera_demo(f_px=500, yaw_deg=0, tx=0.0, tz=0.0):
    Kc = np.array([[f_px, 0, W / 2], [0, f_px, H / 2], [0, 0, 1]], float)
    R = rot_y(yaw_deg)
    t = -R @ np.array([tx, 0, tz])                       # camera centre at (tx, 0, tz) in the world
    Pc = (R @ cube.T).T + t
    p = (Kc @ Pc.T).T
    uv = p[:, :2] / p[:, 2:3]
    fig, ax = plt.subplots(figsize=(7, 5.3))
    for i, j in edges:
        ax.plot(uv[[i, j], 0], uv[[i, j], 1], "o-", color="#1a73e8" if cube[i, 2] == 4.5 and cube[j, 2] == 4.5 else "#9aa0a6", lw=2)
    ax.set_xlim(0, W); ax.set_ylim(H, 0); ax.set_aspect("equal")
    ax.set_title(f"f = {f_px} px, camera at x = {tx:.1f} m, z = {tz:.1f} m, yaw = {yaw_deg} deg")
    plt.show()


interact(camera_demo, f_px=widgets.IntSlider(value=500, min=150, max=1500, step=25),
         yaw_deg=widgets.IntSlider(value=0, min=-30, max=30), tx=widgets.FloatSlider(value=0.0, min=-3, max=3, step=0.1),
         tz=widgets.FloatSlider(value=0.0, min=-10, max=4, step=0.25))

In [ ]:
# Figure 2.1: the same set-up in 3D (drag to rotate)
fig = go.Figure()
for i, j in edges:
    fig.add_trace(go.Scatter3d(x=cube[[i, j], 0], y=cube[[i, j], 2], z=-cube[[i, j], 1], mode="lines",
                               line=dict(color="#1a73e8", width=5), showlegend=False))
depth = 1.0
corners = np.array([[(u - W / 2) / 500 * depth, (v - H / 2) / 500 * depth, depth] for u, v in [(0, 0), (W, 0), (W, H), (0, H)]])
for c in corners:
    fig.add_trace(go.Scatter3d(x=[0, c[0]], y=[0, c[2]], z=[0, -c[1]], mode="lines", line=dict(color="#d93025"), showlegend=False))
loop = np.vstack([corners, corners[:1]])
fig.add_trace(go.Scatter3d(x=loop[:, 0], y=loop[:, 2], z=-loop[:, 1], mode="lines", line=dict(color="#d93025", width=4),
                           name="image plane (f = 1 unit)"))
fig.add_trace(go.Scatter3d(x=[0], y=[0], z=[0], mode="markers", marker=dict(size=6, color="black"), name="optical centre"))
fig.update_layout(title="Figure 2.1 - Camera frustum and the cube in 3D (camera looks along +Z)", height=520,
                  scene=dict(xaxis_title="X", yaxis_title="Z (depth)", zaxis_title="-Y (up)", aspectmode="data"))
fig.show()

## 3. Real lenses: focus and depth of field

A pinhole gives a sharp image at every depth but lets through very little light. A **lens** collects much more light, but focuses
only one depth perfectly. For a thin lens with focal length $f$:

$$ \frac{1}{f} = \frac{1}{d_o} + \frac{1}{d_i}, \qquad \text{magnification } m = \frac{d_i}{d_o} $$

Points at other depths form small blur discs. If a disc is smaller than the acceptable **circle of confusion** $c$ (in machine vision,
about 1–2 pixels), the point still looks sharp. The range of depths that stays sharp is the **depth of field (DoF)**. With the
**f-number** $N = f/D$ (aperture diameter $D$), the hyperfocal distance is $H_f \approx f^2/(Nc)$ and, for a focus distance $s$:

$$ z_\text{near} \approx \frac{s\,H_f}{H_f + s}, \qquad z_\text{far} \approx \frac{s\,H_f}{H_f - s} $$

**Trade-off:** closing the aperture (larger $N$) increases DoF but reduces light (∝ $1/N^2$) and, beyond about f/11, diffraction blurs the image.

In [ ]:
def dof(f_mm, N, s_mm, c_mm):
    Hf = f_mm ** 2 / (N * c_mm) + f_mm
    near = s_mm * (Hf - f_mm) / (Hf + s_mm - 2 * f_mm)
    far = s_mm * (Hf - f_mm) / (Hf - s_mm) if s_mm < Hf else np.inf
    return near, far


fig, axes = plt.subplots(1, 2, figsize=(16, 4.6))
Ns = np.array([1.4, 2, 2.8, 4, 5.6, 8, 11, 16])
for f_mm, col in [(8, "#1a73e8"), (16, "#188038"), (35, "#d93025")]:
    d = [np.subtract(*dof(f_mm, N, 300, 0.007)[::-1]) for N in Ns]
    axes[0].plot(Ns, d, "o-", color=col, label=f"f = {f_mm} mm")
axes[0].set_xscale("log", base=2); axes[0].set_xticks(Ns, [str(n) for n in Ns]); axes[0].set_yscale("log")
axes[0].set_xlabel("f-number N"); axes[0].set_ylabel("depth of field (mm)"); axes[0].grid(alpha=0.3); axes[0].legend()
axes[0].set_title("DoF at 300 mm working distance (c = 7 um, ~2 pixels)")
axes[1].plot(Ns, 100 / Ns ** 2 * 1.4 ** 2, "s-", color="#e37400", label="relative light (1/N^2)")
axes[1].set_xscale("log", base=2); axes[1].set_xticks(Ns, [str(n) for n in Ns]); axes[1].set_ylabel("% of light at f/1.4")
axes[1].axvspan(11, 16, color="#fad2cf", alpha=0.5); axes[1].text(12.5, 60, "diffraction\nlimits sharpness", ha="center")
axes[1].grid(alpha=0.3); axes[1].legend(); axes[1].set_xlabel("f-number N"); axes[1].set_title("the price of DoF: light")
plt.suptitle("Figure 3.1 - Depth of field vs aperture and focal length", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# Figure 3.2: simulated defocus of three identical targets at different depths
target = np.zeros((120, 120), np.uint8)
for k in range(0, 120, 12):
    target[:, k:k + 6] = 255
target[40:80, 40:80] = 255 - target[40:80, 40:80]
depths = [250, 300, 400]


def defocus(img, z, z_focus, aperture):
    """Blur-disc size grows with |1/z - 1/z_focus| and with the aperture diameter (1/N)."""
    blur = abs(1 / z - 1 / z_focus) * 12000 / aperture
    return cv2.GaussianBlur(img, (0, 0), max(blur, 0.01))


fig, axes = plt.subplots(2, 3, figsize=(12, 8))
for r, N in enumerate([2, 8]):
    for c, z in enumerate(depths):
        axes[r, c].imshow(defocus(target, z, 300, N), cmap="gray")
        axes[r, c].set_title(f"target at {z} mm, focus 300 mm, f/{N}"); axes[r, c].axis("off")
plt.suptitle("Figure 3.2 - Only the focused depth is sharp; stopping down (larger N) extends the sharp range", fontweight="bold")
plt.tight_layout(); plt.show()

### Interactive: depth-of-field calculator

In [ ]:
def dof_demo(f_mm=16, N=4.0, focus_mm=300, pixel_um=3.45):
    c = 2 * pixel_um / 1000
    near, far = dof(f_mm, N, focus_mm, c)
    far_txt = "infinity" if np.isinf(far) else f"{far:.0f} mm"
    print(f"Circle of confusion (2 px): {c * 1000:.1f} um")
    print(f"Sharp from {near:.0f} mm to {far_txt}  ->  DoF = {'infinite' if np.isinf(far) else f'{far - near:.1f} mm'}")
    print(f"Relative light vs f/1.4: {(1.4 / N) ** 2 * 100:.0f}%")


interact(dof_demo, f_mm=widgets.IntSlider(value=16, min=4, max=75), N=widgets.FloatSlider(value=4, min=1.4, max=16, step=0.1),
         focus_mm=widgets.IntSlider(value=300, min=50, max=2000, step=10), pixel_um=widgets.FloatSlider(value=3.45, min=1.5, max=10, step=0.05))

## 4. Field of view and choosing a lens

The **field of view** is the size of the scene the sensor covers. For a sensor of width $w_s$ and a lens of focal length $f$, at working
distance $d$ (much larger than $f$):

$$ \text{FOV}_\text{angle} = 2\arctan\frac{w_s}{2f}, \qquad \text{FOV}_\text{width} \approx \frac{w_s\, d}{f}, \qquad
f \approx \frac{w_s\, d}{\text{FOV}_\text{width}} $$

The **spatial resolution** on the object is $\text{FOV}_\text{width} / N_\text{pixels}$ (mm per pixel). A rule of thumb: a feature must
span **at least 3 pixels** to be detected reliably, and measurement tolerance should be **≥ 10×** the pixel size unless you use sub-pixel methods.

| Sensor format (typical) | Width × height (mm) | Example resolution |
|---|---|---|
| 1/2.9" | 4.9 × 3.7 | 1440 × 1080 (1.6 MP) |
| 1/1.8" | 7.1 × 5.3 | 2048 × 1536 (3.2 MP) |
| 2/3" | 8.4 × 7.1 | 2448 × 2048 (5 MP) |
| 1.1" | 14.1 × 10.3 | 4096 × 3000 (12 MP) |

In [ ]:
SENSORS = {'1/2.9" 1.6 MP': (4.9, 1440), '1/1.8" 3.2 MP': (7.1, 2048), '2/3" 5 MP': (8.4, 2448), '1.1" 12 MP': (14.1, 4096)}
STANDARD_LENSES = [4, 6, 8, 12, 16, 25, 35, 50, 75]


def lens_demo(part_width_mm=120, margin_pct=20, working_distance_mm=400, sensor='2/3" 5 MP', smallest_defect_mm=0.3):
    ws, px = SENSORS[sensor]
    fov = part_width_mm * (1 + margin_pct / 100)
    f_ideal = ws * working_distance_mm / fov
    f_std = max([f for f in STANDARD_LENSES if f <= f_ideal], default=STANDARD_LENSES[0])
    fov_real = ws * working_distance_mm / f_std
    mm_px = fov_real / px
    print(f"Required field of view : {fov:.0f} mm")
    print(f"Ideal focal length     : {f_ideal:.1f} mm  -> choose standard {f_std} mm lens (FOV {fov_real:.0f} mm)")
    print(f"Resolution on the part : {mm_px * 1000:.0f} um / pixel")
    print(f"Smallest defect        : {smallest_defect_mm / mm_px:.1f} pixels  "
          f"{'OK (>= 3 px)' if smallest_defect_mm / mm_px >= 3 else 'TOO SMALL: more pixels or a smaller FOV'}")


interact(lens_demo, part_width_mm=widgets.IntSlider(value=120, min=10, max=1000, step=5),
         margin_pct=widgets.IntSlider(value=20, min=0, max=100, step=5),
         working_distance_mm=widgets.IntSlider(value=400, min=50, max=2000, step=10),
         sensor=list(SENSORS), smallest_defect_mm=widgets.FloatSlider(value=0.3, min=0.02, max=5, step=0.01))

## 5. Lens distortion

Real lenses bend rays so that straight lines appear curved. OpenCV uses the Brown–Conrady model on **normalised** coordinates
$x = X/Z,\ y = Y/Z$ with $r^2 = x^2 + y^2$:

$$ x_d = x\,(1 + k_1 r^2 + k_2 r^4 + k_3 r^6) + 2p_1 xy + p_2(r^2 + 2x^2) $$
$$ y_d = y\,(1 + k_1 r^2 + k_2 r^4 + k_3 r^6) + p_1(r^2 + 2y^2) + 2p_2 xy $$

- **Radial** ($k_i$): $k_1 < 0$ gives **barrel** distortion (wide-angle lenses), $k_1 > 0$ **pincushion** (telephoto).
- **Tangential** ($p_i$): the lens is slightly tilted relative to the sensor.

Distortion must be corrected before measuring. Calibration (Session B) estimates these coefficients.
To *simulate* a distorted image we ask, for every pixel of the distorted output, where it came from in the ideal image
(`cv2.undistortPoints` inverts the model), and resample with `cv2.remap`.

In [ ]:
Wd, Hd = 480, 360
Kd = np.array([[360, 0, Wd / 2], [0, 360, Hd / 2], [0, 0, 1]], float)
ideal = np.full((Hd, Wd, 3), 255, np.uint8)
for x in range(0, Wd + 1, 30):
    cv2.line(ideal, (x, 0), (x, Hd), (60, 60, 60), 1)
for y in range(0, Hd + 1, 30):
    cv2.line(ideal, (0, y), (Wd, y), (60, 60, 60), 1)
cv2.rectangle(ideal, (90, 60), (390, 300), (0, 0, 200), 3)


def apply_distortion(img, dist):
    """Render what a lens with coefficients `dist` would show for an ideal image."""
    h, w = img.shape[:2]
    uu, vv = np.meshgrid(np.arange(w, dtype=np.float32), np.arange(h, dtype=np.float32))
    pts = np.stack([uu.ravel(), vv.ravel()], -1)[:, None, :]
    src = cv2.undistortPoints(pts, Kd, np.asarray(dist, float), P=Kd).reshape(h, w, 2)
    return cv2.remap(img, src[..., 0], src[..., 1], cv2.INTER_LINEAR, borderValue=(255, 255, 255))


cases = {"ideal": [0, 0, 0, 0, 0], "barrel  k1 = -0.35": [-0.35, 0.08, 0, 0, 0],
         "pincushion  k1 = +0.35": [0.35, 0, 0, 0, 0], "tangential  p1 = 0.04": [0, 0, 0.04, 0, 0]}
fig, axes = plt.subplots(1, 4, figsize=(21, 4.4))
for ax, (name, dcoef) in zip(axes, cases.items()):
    ax.imshow(cv2.cvtColor(apply_distortion(ideal, dcoef), cv2.COLOR_BGR2RGB)); ax.set_title(name); ax.axis("off")
plt.suptitle("Figure 5.1 - Distortion models applied to a straight grid", fontweight="bold")
plt.tight_layout(); plt.show()

gx, gy = np.meshgrid(np.linspace(0, Wd, 17), np.linspace(0, Hd, 13))
pts = np.stack([gx.ravel(), gy.ravel()], -1)
fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))
for ax, (name, dcoef) in zip(axes, [("barrel", [-0.35, 0.08, 0, 0, 0]), ("tangential", [0, 0, 0.04, 0, 0])]):
    norm = cv2.undistortPoints(pts[:, None, :].astype(np.float32), Kd, None).reshape(-1, 2)
    obj = np.hstack([norm, np.ones((len(norm), 1))])
    dist_px, _ = cv2.projectPoints(obj, np.zeros(3), np.zeros(3), Kd, np.array(dcoef, float))
    d = dist_px.reshape(-1, 2) - pts
    ax.quiver(pts[:, 0], pts[:, 1], d[:, 0], d[:, 1], np.hypot(*d.T), angles="xy", scale_units="xy", scale=0.5, cmap="plasma")
    ax.set_xlim(-20, Wd + 20); ax.set_ylim(Hd + 20, -20); ax.set_aspect("equal")
    ax.set_title(f"{name}: where each pixel moves (arrows x2)")
plt.suptitle("Figure 5.2 - Distortion as a displacement field: radial grows with r^2, tangential is asymmetric", fontweight="bold")
plt.tight_layout(); plt.show()

### Interactive: distortion coefficients

In [ ]:
def distortion_demo(k1=-0.2, k2=0.0, p1=0.0, p2=0.0):
    out = apply_distortion(ideal, [k1, k2, p1, p2, 0])
    show(ideal, out, titles=["ideal", f"k1={k1:.2f} k2={k2:.2f} p1={p1:.3f} p2={p2:.3f}"], figsize=(12, 4.6))


interact(distortion_demo, k1=widgets.FloatSlider(value=-0.2, min=-0.6, max=0.6, step=0.02),
         k2=widgets.FloatSlider(value=0.0, min=-0.3, max=0.3, step=0.02),
         p1=widgets.FloatSlider(value=0.0, min=-0.05, max=0.05, step=0.005),
         p2=widgets.FloatSlider(value=0.0, min=-0.05, max=0.05, step=0.005))

## 6. Vignetting and telecentric lenses

**Vignetting**: image corners are darker because off-axis light arrives obliquely. For an ideal lens the fall-off follows the
**cos⁴ law**, $I(\theta) = I_0 \cos^4\theta$. A 30° off-axis ray already loses 44% of its brightness, so global thresholds fail
near the edges unless you correct with a flat-field image (Week 3).

**Telecentric lenses**: in a normal (entocentric) lens the magnification changes with distance ($m \approx f/(d - f)$), so a
part that moves 5 mm closer looks bigger and a gauge would report the wrong size. A telecentric lens only accepts rays parallel
to the optical axis, so the magnification is **constant over depth**, which is essential for precise gauging. They are large and
expensive, and their field of view can be no bigger than the front lens.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(20, 4.6))
yy, xx = np.mgrid[-1:1:360j, -1.33:1.33:480j]
theta = np.arctan(np.hypot(xx, yy) * np.tan(np.deg2rad(35)) / 1.66)
vign = np.cos(theta) ** 4
flat = cv2.cvtColor(load_sample("sudoku.png"), cv2.COLOR_BGR2GRAY)
flat = cv2.resize(flat, (480, 360)).astype(float) * vign
axes[0].imshow(flat, cmap="gray"); axes[0].axis("off"); axes[0].set_title("image with cos^4 vignetting")
axes[1].plot(np.linspace(0, 40, 100), np.cos(np.deg2rad(np.linspace(0, 40, 100))) ** 4 * 100, lw=2.5)
axes[1].set_xlabel("off-axis angle (deg)"); axes[1].set_ylabel("relative brightness (%)"); axes[1].grid(alpha=0.3)
axes[1].set_title("cos^4 fall-off")
dz = np.linspace(-10, 10, 100)
f_mm, wd = 25, 200
m_ento = f_mm / (wd + dz - f_mm)
axes[2].plot(dz, (m_ento / m_ento[50] - 1) * 100, lw=2.5, label="standard 25 mm lens at 200 mm")
axes[2].plot(dz, np.zeros_like(dz), lw=2.5, label="telecentric lens")
axes[2].set_xlabel("part moves towards (-) / away (+) from camera (mm)"); axes[2].set_ylabel("measured size error (%)")
axes[2].legend(); axes[2].grid(alpha=0.3); axes[2].set_title("why gauging uses telecentric lenses")
plt.suptitle("Figure 6.1 - Vignetting and telecentricity", fontweight="bold")
plt.tight_layout(); plt.show()

## 7. Lighting for machine vision

> *"Lighting is the single most important factor in a machine-vision system."* A defect is only detectable if the lighting makes it
> **contrast** with its surroundings.

| Technique | Geometry | Makes visible | Typical use |
|---|---|---|---|
| **Bright-field** (front, coaxial) | light from (near) the camera direction | colour and print; flat shiny surfaces appear bright | labels, print, flat parts |
| **Dark-field** (low angle, ring) | light grazes the surface at 5–30° | edges, scratches, embossing, dust *glow*; flat areas stay dark | engraving, scratches on glass or metal |
| **Diffuse dome / cloudy-day** | light from all directions | hides glare and relief; shows surface colour | shiny, curved, crinkled packaging |
| **Backlight** | light behind the part | a perfect silhouette | gauging outlines, holes, level checks |
| **Structured light** | projected lines or patterns | 3D shape | height and volume (Week 12) |
| **Colour / wavelength** | coloured or NIR light | same-colour features brighten, complementary ones darken | label contrast, see through dyes |

We simulate a small machined metal part: a raised ring, an engraved lot number, a **scratch**, a **dent** and a printed mark.
Each pixel's brightness follows the **Lambertian** model $I = \rho\, \max(0, \mathbf{n}\cdot\mathbf{l}) + I_\text{ambient}$, where $\rho$ is the
surface albedo (colour), $\mathbf{n}$ the surface normal (from the height map) and $\mathbf{l}$ the light direction.

In [ ]:
n_px = 320
height = np.zeros((n_px, n_px), np.float32)
yy, xx = np.mgrid[0:n_px, 0:n_px]
r = np.hypot(xx - 160, yy - 160)
height += 4.0 * ((r > 70) & (r < 90))                                         # raised ring
text_mask = np.zeros((n_px, n_px), np.uint8)   # OpenCV 5 draws text on 8-bit images only
cv2.putText(text_mask, "LOT 42", (95, 175), cv2.FONT_HERSHEY_SIMPLEX, 1.1, 1, 3)
height -= 1.5 * text_mask                                                       # engraved text
scratch = np.zeros((n_px, n_px), np.uint8); cv2.line(scratch, (30, 270), (290, 235), 1, 2)
height -= 1.2 * scratch                                                         # fine scratch
height -= 3.0 * np.exp(-((xx - 250) ** 2 + (yy - 70) ** 2) / (2 * 7 ** 2))    # dent
height = cv2.GaussianBlur(height, (0, 0), 1.2)
print_mask = np.zeros((n_px, n_px), np.uint8); cv2.rectangle(print_mask, (30, 30), (95, 60), 1, -1)
albedo = np.where(print_mask > 0, 0.25, 0.75).astype(np.float32)             # dark printed mark (colour only, no relief)
outline = np.zeros_like(height, np.uint8)
cv2.rectangle(outline, (10, 10), (310, 310), 1, -1)
cv2.circle(outline, (160, 160), 30, 0, -1); cv2.circle(outline, (60, 250), 12, 0, -1)   # two through-holes

gy_, gx_ = np.gradient(height)
normals = np.dstack([-gx_, -gy_, np.ones_like(height)])
normals /= np.linalg.norm(normals, axis=2, keepdims=True)


def render(elevation_deg, azimuth_deg=0.0, diffuse=False, ambient=0.03):
    if diffuse:                                     # dome: average over many light directions
        img = sum(render(e, a, ambient=0) for e in (30, 60, 85) for a in range(0, 360, 45)) / 24
        return np.clip(img + ambient, 0, 1)
    el, az = np.deg2rad(elevation_deg), np.deg2rad(azimuth_deg)
    light = np.array([np.cos(el) * np.cos(az), np.cos(el) * np.sin(az), np.sin(el)])
    shading = np.clip(normals @ light, 0, None)
    img = albedo * shading + ambient
    return np.clip(img * outline, 0, 1)


views = {"bright-field (light from above)": render(90), "dark-field (grazing 8 deg)": render(8, 20),
         "diffuse dome": render(0, diffuse=True), "backlight (silhouette)": 1 - outline.astype(float)}
fig, axes = plt.subplots(1, 4, figsize=(21, 5.4))
for ax, (name, img) in zip(axes, views.items()):
    ax.imshow(img, cmap="gray", vmin=0, vmax=1); ax.set_title(name); ax.axis("off")
plt.suptitle("Figure 7.1 - The same part under four lighting techniques: each reveals different features", fontweight="bold")
plt.tight_layout(); plt.show()

**Read Figure 7.1 carefully.**
- *Bright-field* shows the printed mark (an albedo change) clearly, but the scratch, dent and engraving are almost invisible.
- *Dark-field* makes every slope glow: the scratch, dent, engraving and ring edges pop out, while the print nearly disappears.
- The *diffuse dome* shows albedo with almost no relief: ideal for reading print on shiny or crumpled surfaces.
- *Backlight* gives a perfect binary silhouette for measuring the outline and the holes, independent of surface finish.

### Interactive: move the lamp
Find the light direction that makes the **scratch** most visible, then the one that hides it. The scratch runs almost horizontally, so which azimuth works best?

In [ ]:
def lighting_demo(elevation_deg=20, azimuth_deg=90):
    img = render(elevation_deg, azimuth_deg)
    fig, axes = plt.subplots(1, 2, figsize=(12, 5), gridspec_kw={"width_ratios": [1.4, 1]})
    axes[0].imshow(img, cmap="gray", vmin=0, vmax=1); axes[0].axis("off")
    axes[0].set_title(f"elevation {elevation_deg} deg, azimuth {azimuth_deg} deg")
    axes[1].plot(img[235:275, 150].ravel() if img.ndim == 2 else [], lw=2)
    axes[1].set_title("brightness profile across the scratch (column 150)"); axes[1].set_xlabel("row 235-275")
    axes[1].set_ylim(0, 1); axes[1].grid(alpha=0.3)
    plt.tight_layout(); plt.show()


interact(lighting_demo, elevation_deg=widgets.IntSlider(value=20, min=2, max=90), azimuth_deg=widgets.IntSlider(value=90, min=0, max=359, step=5))

### Colour of light
A monochrome camera under **red** light sees red objects as bright and green/blue ones as dark; under **green** light the opposite.
Choosing the light colour (or a colour filter on the lens) is a cheap way to maximise contrast. A monochrome camera under a single
colour is approximately one channel of a colour image:

In [ ]:
fruits = load_sample("fruits.jpg")
b, g, r = cv2.split(fruits)
show(fruits, r, g, b, titles=["colour scene", "mono camera + RED light: orange bright, lime dark",
                              "GREEN light: lime brighter", "BLUE light: both fruits dark, cloth bright"], figsize=(21, 4.4))

## 8. Exercises (with self-checks)
Replace each `raise NotImplementedError`, then run the check below it.

### E1. Project world points
Implement the full projection $\lambda[u, v, 1]^\top = K(R\,X_w + t)$ for an `(N, 3)` array of world points. Return `(N, 2)` pixel coordinates.
Do **not** call `cv2.projectPoints`.

In [ ]:
def project_points(Xw, K, R, t):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e1():
    rng = np.random.default_rng(1)
    Xw = rng.uniform([-1, -1, 4], [1, 1, 8], (50, 3))
    rvec = np.array([0.1, -0.2, 0.05]); t = np.array([0.2, -0.1, 0.5])
    R, _ = cv2.Rodrigues(rvec)
    Kt = np.array([[800, 0, 320], [0, 780, 240], [0, 0, 1.0]])
    uv = np.asarray(project_points(Xw, Kt, R, t))
    assert uv.shape == (50, 2), f"expected shape (50, 2), got {uv.shape}"
    ref, _ = cv2.projectPoints(Xw, rvec, t, Kt, None)
    err = np.abs(uv - ref.reshape(-1, 2)).max()
    assert err < 1e-6, f"differs from cv2.projectPoints by up to {err:.3g} px: did you divide by the depth?"


check("E1 project_points", _test_e1)

### E2. Lens selection
Return the focal length (mm) needed so that a sensor of width `sensor_mm` covers exactly `fov_mm` of the scene at `distance_mm`, using the **exact** angular relation (not the approximation).

In [ ]:
def focal_length_for_fov(sensor_mm, distance_mm, fov_mm):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e2():
    for s, d, fov in [(8.4, 400, 150), (4.9, 120, 60), (14.1, 1500, 900)]:
        f = focal_length_for_fov(s, d, fov)
        angle_sensor = 2 * np.arctan(s / (2 * f))
        angle_scene = 2 * np.arctan(fov / (2 * d))
        assert abs(angle_sensor - angle_scene) < 1e-9, f"sensor {s} mm, distance {d} mm, FOV {fov} mm: the angles do not match"


check("E2 focal_length_for_fov", _test_e2)

### E3. The distortion model
Implement the Brown–Conrady model from Section 5 for normalised points `xy` of shape `(N, 2)`, with coefficients `k1, k2, p1, p2` (no $k_3$).

In [ ]:
def distort_normalised(xy, k1, k2, p1, p2):
    # TODO: your code here
    raise NotImplementedError

In [ ]:
def _test_e3():
    rng = np.random.default_rng(2)
    xy = rng.uniform(-0.6, 0.6, (100, 2))
    for coeffs in [(-0.3, 0.1, 0, 0), (0.2, -0.05, 0.01, -0.02), (0, 0, 0.03, 0.02)]:
        out = np.asarray(distort_normalised(xy, *coeffs))
        ref, _ = cv2.projectPoints(np.hstack([xy, np.ones((100, 1))]), np.zeros(3), np.zeros(3), np.eye(3), np.array([*coeffs, 0.0]))
        err = np.abs(out - ref.reshape(-1, 2)).max()
        assert err < 1e-9, f"coefficients {coeffs}: differs from OpenCV by {err:.3g}. Check which term uses p1 and which uses p2."


check("E3 distort_normalised", _test_e3)

### E4. Lighting design (no self-check)
Using `render()`, find settings that make the **dent** clearly visible while keeping the **printed mark** readable in the *same*
image. Is one image enough, or would you capture two images with different lights (a common industrial trick)? Justify.

In [ ]:
# E4: your code here

## 9. Challenge: design a vision system

A factory wants to inspect **bottle caps** (Ø 30 mm) on a conveyor for **scratches ≥ 0.1 mm wide** and **missing printed logos**.
The camera can be mounted **250 mm** above the belt.
1. Choose a sensor from `SENSORS` and a standard lens. Report mm/pixel and how many pixels a 0.1 mm scratch covers.
2. Compute the depth of field at your chosen f-number. Caps may vary in height by ±2 mm: is that inside the DoF?
3. Propose a lighting set-up. Hint: can one light show both scratches *and* print? (Figure 7.1)
4. Present your design as a one-page table with a short justification for each choice.

In [ ]:
# Your design calculations here

## 10. Quick check (auto-marked)

In [ ]:
quiz([
    {
        "q": 'A 2 m tall person is 10 m from a camera with f = 800 px. How tall do they appear in the image?',
        "options": [
            '80 px',
            '160 px',
            '1,600 px',
            '4,000 px',
        ],
        "answer": 'c35f81ea5a',
        "explain": 'y = f Y / Z = 800 x 2 / 10 = 160 pixels.',
    },
    {
        "q": 'What does the intrinsic matrix K contain?',
        "options": [
            "The camera's position and orientation",
            'Focal lengths (in pixels) and the principal point',
            'The lens distortion coefficients',
            'The image resolution only',
        ],
        "answer": '682af8f544',
        "explain": 'Pose is in [R|t]; distortion coefficients are separate.',
    },
    {
        "q": 'Which change increases depth of field?',
        "options": [
            'Opening the aperture (smaller f-number)',
            'Closing the aperture (larger f-number)',
            'Using a longer focal length at the same distance',
            'Moving the camera closer',
        ],
        "answer": 'b2492c7401',
        "explain": 'Smaller apertures make smaller blur discs, at the cost of light and, eventually, diffraction.',
    },
    {
        "q": 'A wide-angle lens makes straight lines near the edges bow outwards. What is this?',
        "options": [
            'Pincushion distortion (k1 > 0)',
            'Barrel distortion (k1 < 0)',
            'Tangential distortion',
            'Vignetting',
        ],
        "answer": 'fb388e31ab',
        "explain": 'Magnification decreases with distance from the centre.',
    },
    {
        "q": 'Why do precision gauging systems use telecentric lenses?',
        "options": [
            'They are cheaper',
            'Magnification does not change when the part moves along the optical axis',
            'They give a wider field of view',
            'They remove the need for lighting',
        ],
        "answer": '6b139af747',
        "explain": 'With a normal lens, a part 5 mm closer looks larger.',
    },
    {
        "q": 'You must find fine scratches on a polished flat metal surface. Which lighting is the best first choice?',
        "options": [
            'Backlight',
            'Dark-field (low-angle) lighting',
            'Diffuse dome',
            'Bright-field coaxial light',
        ],
        "answer": '79c935519c',
        "explain": 'Grazing light is reflected towards the camera only by slopes such as scratch edges, so they glow on a dark background.',
    },
    {
        "q": 'A red label must be read with a monochrome camera. Which light colour gives the red print the MOST contrast against a white background?',
        "options": [
            'Red light',
            'Green or blue light',
            'Infrared light',
            'White light',
        ],
        "answer": '73708ae1cf',
        "explain": 'Under green/blue light the red ink reflects little and looks dark, while the white background stays bright.',
    },
], title='Week 02A quiz')

## 11. Exit ticket (reflection)
1. Explain in one sentence each: why does zooming differ from walking closer?
2. Your gauging system reports parts 1% too large whenever the conveyor vibrates vertically. What is happening, and what would you change?
3. Which single lighting technique would you choose to check whether all holes in a sheet-metal part were punched? Why?

## 12. What's new (2025–26)

> - **Liquid lenses** refocus electronically in milliseconds, so one camera can inspect parts of different heights without mechanical focusing.
> - **SWIR cameras** (short-wave infrared, 900–1700 nm) are dropping in price; they see moisture, through silicon and plastics, and separate materials that look identical in visible light.
> - **Computational illumination**: multi-light "photometric stereo" systems capture several images under different lights and compute the surface normals directly (Week 12), making scratches and dents visible regardless of colour.
> - **Event cameras** and high-speed global-shutter sensors remove motion blur on fast lines without strobe lighting.

## 13. References and further exploration

**Textbooks**
- Hartley, R. & Zisserman, A. (2004). *Multiple View Geometry in Computer Vision*, 2nd ed., ch. 6 (camera models). Cambridge University Press.
- Szeliski (2022). *Computer Vision: Algorithms and Applications*, 2nd ed., §2.1 (geometric image formation), §2.2 (photometric image formation). [szeliski.org/Book](https://szeliski.org/Book/)
- Steger, Ulrich & Wiedemann (2018). *Machine Vision Algorithms and Applications*, 2nd ed., ch. 2 (illumination, lenses, cameras). Wiley.
- Hecht, E. (2017). *Optics*, 5th ed., ch. 5–6 (geometrical optics). Pearson.

**Papers and standards**
- Brown, D. C. (1966). *Decentering distortion of lenses.* Photogrammetric Engineering 32(3).
- Conrady, A. E. (1919). *Decentred lens-systems.* Monthly Notices of the Royal Astronomical Society 79(5). [doi:10.1093/mnras/79.5.384](https://doi.org/10.1093/mnras/79.5.384)

**Guides, videos and interactive**
- Edmund Optics: *Imaging Resource Guide* (lens selection, depth of field, telecentricity). [edmundoptics.com](https://www.edmundoptics.com/knowledge-center/application-notes/imaging/)
- Nayar, S. *First Principles of Computer Vision*: "Image Formation" lectures (pinhole, lenses, depth of field). [fpcv.cs.columbia.edu](https://fpcv.cs.columbia.edu/)
- OpenCV camera model documentation. [docs.opencv.org](https://docs.opencv.org/4.x/d9/d0c/group__calib3d.html)
- Wikipedia: *Pinhole camera model*, *Depth of field*, *Distortion (optics)*, *Telecentric lens*.

**Images:** `sudoku.png`, `fruits.jpg` from the OpenCV sample data; the machined part, grid and 3D scenes are generated in code.

## 14. Key takeaways
- Pinhole projection: $x = fX/Z$. In pixels, $\lambda \mathbf{u} = K[R\,|\,\mathbf{t}]\mathbf{X}$. Intrinsics describe the camera; extrinsics its pose.
- Perspective: size ∝ 1/Z and parallel lines meet at vanishing points.
- Lenses trade light for **depth of field** (f-number). Pick $f \approx w_s d / \text{FOV}$ and check that features span ≥ 3 pixels.
- Real lenses add **distortion** (barrel/pincushion/tangential) and **vignetting**; **telecentric** lenses keep magnification constant for gauging.
- **Lighting creates contrast**: bright-field for print, dark-field for scratches and relief, dome for shiny parts, backlight for outlines, colour to separate colours.

**Next session (02B):** calibrate a real camera from chessboard images, undistort, estimate pose and measure in real units.